# 01 — Data Understanding

This notebook performs a read-only structural review of the LendingClub source files. It does not clean data, remove columns, engineer features, or build predictive models. CSV files are profiled in chunks to support large files.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data.load_data import profile_csv, raw_file_status, serialise_status
from utils.helpers import (
    classify_column_name,
    format_bytes,
    leakage_screen,
    target_candidate_screen,
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
CHUNK_SIZE = 100_000
RAW_DIR

## 1. Source-file availability

The expected files must be present before results can be interpreted. This check reads only file metadata.

In [ ]:
status = pd.DataFrame(serialise_status(raw_file_status(RAW_DIR)))
status['file_size'] = status['size_bytes'].map(format_bytes)
display(status[['name', 'exists', 'file_size', 'path']])

available_csvs = [
    RAW_DIR / name
    for name in ('accepted_2007_to_2020Q3.csv', 'rejected_2007_to_2020Q3.csv')
    if (RAW_DIR / name).is_file()
]
if not available_csvs:
    display(Markdown('> **Source data unavailable:** place the required LendingClub files in `data/raw/`, then re-run this notebook.'))

## 2. Data dictionary overview

The workbook is inspected read-only to identify sheets and display a small preview of each sheet.

In [ ]:
dictionary_path = RAW_DIR / 'LendingClubDataDictionary.xlsx'
if dictionary_path.is_file():
    workbook = pd.ExcelFile(dictionary_path)
    display(pd.DataFrame({'sheet_name': workbook.sheet_names}))
    for sheet_name in workbook.sheet_names:
        display(Markdown(f'### {sheet_name}'))
        display(pd.read_excel(dictionary_path, sheet_name=sheet_name, nrows=10))
else:
    display(Markdown('> Data dictionary is not available yet.'))

## 3. Chunked CSV structural profiles

The following cells calculate row counts, columns, inferred memory, missingness, data types, and samples without modifying source files. Runtime will depend on file size.

In [ ]:
profiles = {}
for csv_path in available_csvs:
    display(Markdown(f'## Profiling `{csv_path.name}`'))
    profiles[csv_path.name] = profile_csv(csv_path, chunksize=CHUNK_SIZE)

overview = pd.DataFrame([
    {
        'file': name,
        'rows': profile['rows'],
        'columns': profile['columns'],
        'file_size': format_bytes(profile['file_size_bytes']),
        'estimated_loaded_memory': format_bytes(profile['estimated_memory_bytes']),
    }
    for name, profile in profiles.items()
])
display(overview)

In [ ]:
for name, profile in profiles.items():
    display(Markdown(f'### Variable profile — `{name}`'))
    variable_summary = pd.DataFrame(profile['column_profile'])
    display(variable_summary)
    display(Markdown('#### Highest missingness'))
    display(variable_summary.sort_values('missing_pct', ascending=False).head(25))
    display(Markdown('#### Sample records'))
    display(pd.DataFrame(profile['sample_records']))

## 4. Initial column grouping and temporal-risk screen

These groupings are screening outputs only. Final business definitions and all application-time eligibility decisions must be confirmed against the data dictionary and the documented decision timestamp.

In [ ]:
for name, profile in profiles.items():
    columns = [row['column'] for row in profile['column_profile']]
    grouping = pd.DataFrame({
        'column': columns,
        'initial_classification': [classify_column_name(column) for column in columns],
    })
    display(Markdown(f'### Initial grouping — `{name}`'))
    display(grouping)
    display(Markdown('#### Candidate target fields for definition review'))
    display(pd.DataFrame({'column': target_candidate_screen(columns)}))
    display(Markdown('#### Fields requiring leakage review before application modelling'))
    display(pd.DataFrame({'column': leakage_screen(columns)}))